# Vision Station - train models on Colab

For a line PC with no GPU. Train here on Colab's free GPU, download the result,
drop it into the project's `models` folder. Inference on the line PC needs no GPU.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**.

What you train here is **a YOLO detector** from images you labelled in
Vision Station's Label workspace -- screws, connectors, components, defects.
It is exported to ONNX, which Vision Station runs with onnxruntime (on the GPU
with the onnxruntime-directml build) or OpenVINO.

Text reading needs no training: Read text and Verify text use the PP-OCRv6
networks (`python get_ocr_models.py`).


In [ ]:
import torch, subprocess
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE - switch the runtime to T4 GPU before training")


## Part 1 - YOLO detector from your labelled images

In Vision Station: **Label** workspace, label your images, then
**Dataset -> Export for training**. Zip the exported folder and upload it below.


In [ ]:
!pip -q install ultralytics openvino
from google.colab import files
up = files.upload()                 # choose your <model>_dataset.zip
zipname = next(iter(up))
!unzip -q -o "$zipname" -d /content/dataset
import glob, os
yaml = glob.glob("/content/dataset/**/data.yaml", recursive=True)[0]
print("dataset:", yaml)


The `data.yaml` paths were written for your PC. Point them at Colab instead.


In [ ]:
import yaml as Y
root = os.path.dirname(yaml)
cfg = Y.safe_load(open(yaml))
cfg["path"] = root
Y.safe_dump(cfg, open(yaml, "w"))
print(cfg)


Train. `yolov8n` is the right default for a line PC: small enough to run fast on
CPU. Move to `yolov8s` only if nano is not accurate enough after adding data --
more labelled images helps far more than a bigger model.

Augmentation is set for inspection, not for photos of the world: no horizontal
flip, because a mirrored board is a different part; modest rotation, because a
fixture only allows a little; strong brightness variation, because lighting
drift is the usual enemy on a real line.


In [ ]:
from ultralytics import YOLO
m = YOLO("yolov8n.pt")
m.train(data=yaml, epochs=150, imgsz=640, batch=16, patience=40,
        degrees=8.0, translate=0.08, scale=0.25, shear=2.0,
        fliplr=0.0, flipud=0.0, hsv_h=0.01, hsv_s=0.45, hsv_v=0.35,
        mosaic=0.7, close_mosaic=15, project="/content/runs", name="yolo")
metrics = m.val()
print("mAP50    %.4f" % metrics.box.map50)
print("mAP50-95 %.4f" % metrics.box.map)


Read mAP50 like this:

| mAP50 | meaning |
|---|---|
| above 0.95 | ready for the line |
| 0.85 - 0.95 | usable, expect occasional misses |
| below 0.85 | label more images - do not train longer |


In [ ]:
best = "/content/runs/yolo/weights/best.pt"
onnx_path = YOLO(best).export(format="onnx", opset=12, simplify=True, dynamic=False)
print("ONNX:", onnx_path)
files.download(onnx_path)


Put the downloaded `best.onnx` in `projects/<your model>/models/`, then add a
**Detect with YOLO** tool and set its model file. Class names travel inside the
file, so you do not retype them.


## Choosing precision on the line PC

After copying a model across, run `python bench_inference.py` **on the line PC
itself**. OpenVINO's fast mode uses reduced precision, and how much that buys
depends on the CPU: recent Xeons and some recent Core processors have hardware
for it, older ones gain little. The benchmark also reports whether fast mode
moves any detections on your own images -- only switch to it if they do not.
